# *Modeling*

Following a comprehensive Exploratory Data Analysis (EDA) and visual feature evaluation, we now proceed to the model implementation phase. Given the binary nature of our target variable (dropout vs. retention), we will establish a Binary Logistic Regression model as our predictive baseline.

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

### ***Data Preparation***

In [16]:
PATH = "../data/processed/dataset.csv"
df = pd.read_csv(PATH)

In [17]:
# Let's instantiate our training variables

X = df.drop("Target", axis= 1)
Y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size= 0.2,
    stratify= Y,
    random_state= 47
)


### ***Baseline model***

In [18]:
# let's create our model!

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegressionCV(
        cv= 5,
        random_state=42,
        max_iter=1000,
        n_jobs=-1,
        solver="lbfgs",
        class_weight="balanced",
        scoring="roc_auc",
    ))
])

model.fit(X_train, y_train)

/home/angel/Programacion/ml/student_dropout/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/angel/Programacion/ml/student_dropout/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](34,)","['Marital status','Application mode','Application order',..., 'Unemployment rate','Inflation rate','GDP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,34
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


#### ***Evaluation***

In [19]:
metrics = model.score(X_test, y_test)
print(f"Metrics with standarization: {(metrics)*100:.4}%")

Metrics with standarization: 91.65%


In [20]:
# Metrics

#Confusion matrixx
y_pred = model.predict(X_test)

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

# Classification report

print("\n Classification report")
print(classification_report(y_test, y_pred))

Confusion Matrix
[[547  54]
 [ 51 233]]

 Classification report
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       601
           1       0.81      0.82      0.82       284

    accuracy                           0.88       885
   macro avg       0.86      0.87      0.86       885
weighted avg       0.88      0.88      0.88       885



The Logistic Regression model yields strong baseline performance, achieving an overall accuracy of 88% and a recall score of 0.82 for the dropout class. These metrics demonstrate the model's robust capability to effectively identify students at risk of attrition before they drop out

#### **Threshold analysis:**

Lowering the classification threshold was evaluated to reduce false negatives. However, the reduction in false negatives was relatively small compared with the increase in false positives. Therefore, the default threshold of 0.5 was retained.

### ***Coefficient Evaluation***

In [21]:
# Sort features by absolute weight (largest impact first)
feature_weights = pd.DataFrame({
    "feature": X_train.columns,
    "weight": model.named_steps["classifier"].coef_[0]
})

feature_weights_sorted = (
    feature_weights
    .assign(abs_weight=feature_weights["weight"].abs())
    .sort_values("abs_weight", ascending=False)
    .drop(columns="abs_weight")
    .reset_index(drop=True)
)

feature_weights_sorted

,feature,weight
0,Curricular units 2nd sem (approved),-1.894215
1,Curricular units 1st sem (approved),-0.903357
2,Curricular units 2nd sem (enrolled),0.830224
3,Tuition fees up to date,-0.796779
4,Curricular units 1st sem (credited),0.457166
5,International,-0.369724
6,Curricular units 2nd sem (grade),-0.344650
7,Course,0.343912
8,Age at enrollment,0.300890
9,Nacionality,0.255736


The model coefficients represent the log-odds contribution of each feature to the target variable. Based on the results, 'International' carries the largest negative coefficient, closely followed by 'Tuition fees up to date' and 'Curricular units 2nd sem (approved)'. In a logistic regression framework, these substantial negative weights indicate that these features are the strongest predictors for reducing dropout risk, heavily contributing toward student retention.

However, it is critical to note that the 'International' feature is present in only approximately 2.5% of the total cases. Given this low variance and high class imbalance, its substantial negative coefficient should be interpreted with caution, as it may lack generalizability across the broader student population.

### ***Conclusions***

A Logistic Regression model integrated with k-fold Cross-Validation was trained to predict student attrition. To address potential class imbalance, a balanced class weight strategy was deployed alongside the standard solver.

This optimization achieved a 82% recall score for the target (dropout) class. While lowering the classification threshold below 0.5 was evaluated to capture more false negatives, the adjustment was ultimately rejected; the marginal reduction in false negatives did not justify the substantial inflation of false positives.

Furthermore, the model's coefficients were analyzed to understand feature contribution and feature importance. It is crucial to note that these weights strictly quantify predictive correlation within the framework and do not imply causal relationships. Overall, the baseline model demonstrates a robust macro performance metric of 88%.

(Optional) Finally, if you'd like to save any other change you've applied. You only have to execute this final code Cell

In [22]:
import joblib

joblib.dump(model, "../models/logistic_regression_cv.pkl")

['../models/logistic_regression_cv.pkl']